# Notebook 01 — Compiler Baselines

This notebook reproduces the **SABRE baseline** results (Table 4, column *Baseline (SABRE)*) and the **Mapper-only** ablation column using the hardware-aware SA+ILP compilation pass (§4.1).  
Results are written to `data/results/compiler_baselines.csv`.

**What is measured**
- Algorithm success probability `S` for each benchmark circuit  
- Post-selection retention `R`  
- End-to-end compilation latency  
- 95 % bootstrap confidence intervals (B = 10 000 resamples)

In [ ]:
import sys, warnings
from pathlib import Path

warnings.filterwarnings('ignore')
ROOT = Path('..').resolve()
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

RESULTS_DIR = ROOT / 'data' / 'results'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
# ── load benchmark circuits ───────────────────────────────────────────────────
from qiskit import QuantumCircuit

QASM_DIR = ROOT / 'circuits'

BENCHMARKS = {
    'VQE-H2':   str(QASM_DIR / 'vqe_h2.qasm'),
    'VQE-LiH':  str(QASM_DIR / 'vqe_lih.qasm'),
    'QPE-12':   str(QASM_DIR / 'phase_est_12.qasm'),
    'Grover-10':str(QASM_DIR / 'grover_10.qasm'),
}

circuits = {}
for name, path in BENCHMARKS.items():
    try:
        circuits[name] = QuantumCircuit.from_qasm_file(path)
        print(f'  loaded {name}: {circuits[name].num_qubits} qubits, depth {circuits[name].depth()}')
    except Exception as e:
        print(f'  ✗ {name}: {e}')

In [ ]:
# ── noise models ──────────────────────────────────────────────────────────────
from src.simulation.noise_models import (
    build_superconducting_noise_model,
    build_trapped_ion_noise_model,
    build_adversarial_noise_model,
)

NOISE_PROFILES = {
    'superconducting': build_superconducting_noise_model(),
    'trapped_ion':     build_trapped_ion_noise_model(),
    'adversarial':     build_adversarial_noise_model(),
}

In [ ]:
# ── SABRE baseline compilation ────────────────────────────────────────────────
from qiskit import transpile
from qiskit_aer import AerSimulator
from src.metrics.aggregator import compute_success_probability, compute_retention
from src.metrics.statistical_tests import bootstrap_ci
from src.simulation.simulator import run_noisy_simulation

N_TRIALS = 100  # paper: ≥100 independent trials (seed 42)
SHOTS    = 8192
B_BOOT   = 10_000


def sabre_compile(qc: QuantumCircuit, n_qubits: int) -> QuantumCircuit:
    """Transpile circuit using SABRE routing (optimisation level 1)."""
    return transpile(
        qc,
        basis_gates=['cx', 'u1', 'u2', 'u3', 'id'],
        optimization_level=1,
        routing_method='sabre',
        layout_method='sabre',
        seed_transpiler=42,
    )


print('SABRE compilation helper ready.')

In [ ]:
# ── hardware-aware mapper ─────────────────────────────────────────────────────
from src.compiler.mapping_pass import HardwareAwareMappingPass
import time

rows = []

for circ_name, qc in tqdm(circuits.items(), desc='Benchmarks'):
    n_q = qc.num_qubits
    noise_model = NOISE_PROFILES['superconducting']  # primary profile

    # ---- SABRE baseline ----
    t0 = time.perf_counter()
    compiled_sabre = sabre_compile(qc, n_q)
    lat_sabre = (time.perf_counter() - t0) * 1e3  # ms

    s_sabre_trials, r_sabre_trials = [], []
    for trial in range(N_TRIALS):
        counts = run_noisy_simulation(compiled_sabre, noise_model, shots=SHOTS, seed=trial)
        s_sabre_trials.append(compute_success_probability(counts))
        r_sabre_trials.append(compute_retention(counts, shots=SHOTS))

    s_sabre_mean = float(np.mean(s_sabre_trials))
    r_sabre_mean = float(np.mean(r_sabre_trials))
    s_sabre_ci   = bootstrap_ci(s_sabre_trials, B=B_BOOT)

    # ---- Mapper-only (SA+ILP) ----
    t1 = time.perf_counter()
    mapper = HardwareAwareMappingPass(n_qubits=n_q)
    compiled_hw = mapper.run(qc)
    lat_hw = (time.perf_counter() - t1) * 1e3

    s_hw_trials, r_hw_trials = [], []
    for trial in range(N_TRIALS):
        counts = run_noisy_simulation(compiled_hw, noise_model, shots=SHOTS, seed=trial)
        s_hw_trials.append(compute_success_probability(counts))
        r_hw_trials.append(compute_retention(counts, shots=SHOTS))

    s_hw_mean = float(np.mean(s_hw_trials))
    r_hw_mean = float(np.mean(r_hw_trials))
    s_hw_ci   = bootstrap_ci(s_hw_trials, B=B_BOOT)

    rows.append({
        'circuit':         circ_name,
        'n_qubits':        n_q,
        'depth':           qc.depth(),
        # SABRE
        'sabre_S_mean':    round(s_sabre_mean, 4),
        'sabre_S_ci_lo':   round(s_sabre_ci[0], 4),
        'sabre_S_ci_hi':   round(s_sabre_ci[1], 4),
        'sabre_R_mean':    round(r_sabre_mean, 4),
        'sabre_lat_ms':    round(lat_sabre, 1),
        # Mapper-only
        'mapper_S_mean':   round(s_hw_mean, 4),
        'mapper_S_ci_lo':  round(s_hw_ci[0], 4),
        'mapper_S_ci_hi':  round(s_hw_ci[1], 4),
        'mapper_R_mean':   round(r_hw_mean, 4),
        'mapper_lat_ms':   round(lat_hw, 1),
    })

df_baseline = pd.DataFrame(rows)
out_path = RESULTS_DIR / 'compiler_baselines.csv'
df_baseline.to_csv(out_path, index=False)
print(f'\nSaved: {out_path}')
df_baseline

In [ ]:
# ── quick bar-chart comparison ────────────────────────────────────────────────
import matplotlib.pyplot as plt
import numpy as np

x      = np.arange(len(df_baseline))
width  = 0.35
labels = df_baseline['circuit'].tolist()

fig, ax = plt.subplots(figsize=(9, 4))
bars1 = ax.bar(x - width/2, df_baseline['sabre_S_mean'],   width, label='SABRE baseline',   color='#aaaaaa')
bars2 = ax.bar(x + width/2, df_baseline['mapper_S_mean'],  width, label='Mapper-only (SA+ILP)', color='#f4a261')

ax.set_xticks(x)
ax.set_xticklabels(labels)
ax.set_ylabel('Mean success probability  S')
ax.set_title('Compiler Baseline vs Hardware-Aware Mapper (Superconducting profile)')
ax.legend()
ax.set_ylim(0, 0.9)
ax.grid(axis='y', alpha=0.35)
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'compiler_baselines.png', dpi=150)
plt.show()

In [ ]:
# ── cross-profile scan ────────────────────────────────────────────────────────
print('\nCross-profile SABRE results for VQE-H2:')
if 'VQE-H2' in circuits:
    qc_h2 = circuits['VQE-H2']
    compiled_sabre_h2 = sabre_compile(qc_h2, qc_h2.num_qubits)
    for pname, nm in NOISE_PROFILES.items():
        s_vals = []
        for t in range(50):
            counts = run_noisy_simulation(compiled_sabre_h2, nm, shots=SHOTS, seed=t)
            s_vals.append(compute_success_probability(counts))
        print(f'  {pname:20s}  S = {np.mean(s_vals):.3f}')